In [1]:
import pandas as pd
import numpy as np
import os

KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'
RECBOLE_DIR = r'D:\Memoire\LLMKGER\recbole_data\llmkger'
os.makedirs(RECBOLE_DIR, exist_ok=True)
np.random.seed(42)

# Charger toutes les interactions
train = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
valid = pd.read_csv(os.path.join(KG_DIR, 'interactions_valid.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
test  = pd.read_csv(os.path.join(KG_DIR, 'interactions_test.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
all_inter = pd.concat([train, valid, test], ignore_index=True)
print(f'Interactions totales : {len(all_inter):,}')
print(f'Users : {all_inter["user_id"].nunique():,}')


Interactions totales : 5,258,360
Users : 106,735


In [2]:
# ── Paramètres proches E-KGAT ─────────────────────────────────────────────
MIN_INTER = 5     # minimum 5 interactions (comme MOOCs réels)
MAX_INTER = 40    # plafonner pour densité réaliste
MAX_USERS = 40000 # proche ordre de grandeur Yasmine

user_counts = all_inter.groupby('user_id').size()
valid_users = user_counts[(user_counts >= MIN_INTER) & (user_counts <= MAX_INTER)].index
print(f'Users avec {MIN_INTER}-{MAX_INTER} interactions : {len(valid_users):,}')

if len(valid_users) > MAX_USERS:
    sampled_users = np.random.choice(valid_users, MAX_USERS, replace=False)
else:
    sampled_users = valid_users

filtered = all_inter[all_inter['user_id'].isin(sampled_users)].copy()

print(f'\n=== Dataset final ===')
print(f'  Interactions : {len(filtered):,}')
print(f'  Users : {filtered["user_id"].nunique():,}')
print(f'  Items : {filtered["item_id"].nunique():,}')
print(f'  Moyenne : {filtered.groupby("user_id").size().mean():.1f} interactions/user')
print(f'\n  Comparaison E-KGAT : 210,456 users, 1,347,618 interactions, 6.4/user')


Users avec 5-40 interactions : 27,553

=== Dataset final ===
  Interactions : 758,705
  Users : 27,553
  Items : 2,808
  Moyenne : 27.5 interactions/user

  Comparaison E-KGAT : 210,456 users, 1,347,618 interactions, 6.4/user


In [3]:
# ── Écrire le fichier .inter ──────────────────────────────────────────────
out = filtered[['user_id', 'item_id']].copy()
out['rating'] = 1.0
out.columns = ['user_id:token', 'item_id:token', 'rating:float']

inter_path = os.path.join(RECBOLE_DIR, 'llmkger.inter')
out.to_csv(inter_path, sep='\t', index=False)
size = os.path.getsize(inter_path) / 1024 / 1024
print(f'✅ .inter : {len(out):,} interactions, {size:.1f} MB')

# ── Recréer .kg et .link ──────────────────────────────────────────────────
df_kg = pd.read_csv(os.path.join(KG_DIR, 'kg_train.tsv'),
                    sep='\t', header=0, names=['head','relation','tail'])
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()
kg_out = df_kg_sem.copy()
kg_out.columns = ['head_id:token', 'relation_id:token', 'tail_id:token']
kg_out.to_csv(os.path.join(RECBOLE_DIR, 'llmkger.kg'), sep='\t', index=False)
print(f'✅ .kg : {len(kg_out):,} triplets')

items_kept = set(filtered['item_id'].unique())
link_out = pd.DataFrame({'item_id:token': list(items_kept),
                          'entity_id:token': list(items_kept)})
link_out.to_csv(os.path.join(RECBOLE_DIR, 'llmkger.link'), sep='\t', index=False)
print(f'✅ .link : {len(link_out):,} liens')


✅ .inter : 758,705 interactions, 18.0 MB
✅ .kg : 648,349 triplets
✅ .link : 2,808 liens


In [4]:
# ── Supprimer le cache RecBole (.pth) ─────────────────────────────────────
import glob
for f in glob.glob(os.path.join(RECBOLE_DIR, '*.pth')):
    os.remove(f)
    print(f'Cache supprimé : {os.path.basename(f)}')

print('\nContenu du dossier :')
for f in os.listdir(RECBOLE_DIR):
    sz = os.path.getsize(os.path.join(RECBOLE_DIR, f)) / 1024 / 1024
    print(f'  {f} : {sz:.1f} MB')



Contenu du dossier :
  llmkger.inter : 18.0 MB
  llmkger.kg : 40.4 MB
  llmkger.link : 0.1 MB


In [7]:
import os, glob, shutil

RECBOLE_DIR = r'D:\Memoire\LLMKGER\recbole_data\llmkger'

# 1. Supprimer le cache dataset .pth
for f in glob.glob(os.path.join(RECBOLE_DIR, '*.pth')):
    os.remove(f)
    print(f'Cache supprimé : {os.path.basename(f)}')

# 2. Supprimer les dossiers log et saved de RecBole (dans le répertoire courant)
for folder in ['log', 'saved', 'log_tensorboard']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f'Dossier supprimé : {folder}')

print('\n✅ Nettoyage terminé')

# Vérifier ce qui reste dans recbole_data
print('\nFichiers dataset restants :')
for f in os.listdir(RECBOLE_DIR):
    sz = os.path.getsize(os.path.join(RECBOLE_DIR, f)) / 1024 / 1024
    print(f'  {f} : {sz:.1f} MB')

Dossier supprimé : log
Dossier supprimé : saved
Dossier supprimé : log_tensorboard

✅ Nettoyage terminé

Fichiers dataset restants :
  llmkger.inter : 18.0 MB
  llmkger.kg : 40.4 MB
  llmkger.link : 0.1 MB


In [11]:
import torch

# Patch torch.load pour RecBole (PyTorch 2.6 weights_only)
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load

print('torch.load patché ✅ (weights_only=False par défaut)')

torch.load patché ✅ (weights_only=False par défaut)


In [12]:
import os, json
import torch
from logging import getLogger
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model, get_trainer

def train_model(model_name, config_dict):
    config = Config(model=model_name, dataset='llmkger', config_dict=config_dict)
    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model = get_model(config['model'])(config, train_data._dataset).to(config['device'])
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)
    # show_progress=False → évite que VS Code rame avec les barres tqdm
    trainer.fit(train_data, valid_data, saved=True, show_progress=False)
    return trainer.evaluate(test_data, show_progress=False)

print('train_model défini (sans barres progression) ✅')

train_model défini (sans barres progression) ✅


In [13]:
RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data'
base_config = {
    'data_path'     : RECBOLE_ROOT,
    'dataset'       : 'llmkger',
    'USER_ID_FIELD' : 'user_id',
    'ITEM_ID_FIELD' : 'item_id',
    'RATING_FIELD'  : 'rating',
    'load_col'      : {'inter': ['user_id', 'item_id', 'rating']},
    'eval_args': {'split': {'LS': 'valid_and_test'}, 'order': 'RO',
                  'group_by': 'user', 'mode': 'full'},
    'metrics'      : ['Recall', 'NDCG', 'Precision', 'Hit', 'MRR'],
    'topk'         : [5, 10],
    'valid_metric' : 'NDCG@10',
    'epochs'           : 30,
    'train_batch_size' : 2048,
    'eval_batch_size'  : 2048,
    'learning_rate'    : 0.001,
    'embedding_size'   : 64,
    'stopping_step'    : 5,
    'device'           : 'cuda',
    'seed'             : 42,
    'show_progress'    : False,
    'verbose'          : False,
}
print('Config définie ✅')

Config définie ✅


In [14]:
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'
os.makedirs(RESULTS_REC, exist_ok=True)

print('=== Entraînement BPRMF ===')
test_result_bpr = train_model('BPR', base_config)

print('\nRésultats BPRMF :')
for k, v in test_result_bpr.items():
    print(f'  {k} : {v:.4f}')

with open(os.path.join(RESULTS_REC, 'BPRMF_results.json'), 'w') as f:
    json.dump({k: float(v) for k, v in test_result_bpr.items()}, f, indent=2)
print('\nBPRMF sauvegardé ✅')

=== Entraînement BPRMF ===


c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett


Résultats BPRMF :
  recall@5 : 0.1540
  recall@10 : 0.2292
  ndcg@5 : 0.1049
  ndcg@10 : 0.1291
  precision@5 : 0.0308
  precision@10 : 0.0229
  hit@5 : 0.1540
  hit@10 : 0.2292
  mrr@5 : 0.0888
  mrr@10 : 0.0988

BPRMF sauvegardé ✅


In [16]:
import scipy.sparse

# Patch pour NGCF : réajouter _update à dok_matrix (supprimé dans scipy récent)
if not hasattr(scipy.sparse.dok_matrix, '_update'):
    def _update(self, data):
        for key, value in data.items():
            dict.__setitem__(self, key, value)
    scipy.sparse.dok_matrix._update = _update
    print('Patch dok_matrix._update appliqué ✅')
else:
    print('dok_matrix._update existe déjà')

Patch dok_matrix._update appliqué ✅


In [17]:
print('=== Entraînement NGCF ===')
test_result_ngcf = train_model('NGCF', {**base_config, 'n_layers': 3})

print('\nRésultats NGCF :')
for k, v in test_result_ngcf.items():
    print(f'  {k} : {v:.4f}')

with open(os.path.join(RESULTS_REC, 'NGCF_results.json'), 'w') as f:
    json.dump({k: float(v) for k, v in test_result_ngcf.items()}, f, indent=2)
print('\nNGCF sauvegardé ✅')

=== Entraînement NGCF ===


c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett


Résultats NGCF :
  recall@5 : 0.1268
  recall@10 : 0.1942
  ndcg@5 : 0.0855
  ndcg@10 : 0.1071
  precision@5 : 0.0254
  precision@10 : 0.0194
  hit@5 : 0.1268
  hit@10 : 0.1942
  mrr@5 : 0.0720
  mrr@10 : 0.0808

NGCF sauvegardé ✅


In [18]:
print('=== Entraînement LightGCN ===')
test_result_lgcn = train_model('LightGCN', {**base_config, 'n_layers': 3})

print('\nRésultats LightGCN :')
for k, v in test_result_lgcn.items():
    print(f'  {k} : {v:.4f}')

with open(os.path.join(RESULTS_REC, 'LightGCN_results.json'), 'w') as f:
    json.dump({k: float(v) for k, v in test_result_lgcn.items()}, f, indent=2)
print('\nLightGCN sauvegardé ✅')

=== Entraînement LightGCN ===


c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett


Résultats LightGCN :
  recall@5 : 0.0883
  recall@10 : 0.1408
  ndcg@5 : 0.0588
  ndcg@10 : 0.0756
  precision@5 : 0.0177
  precision@10 : 0.0141
  hit@5 : 0.0883
  hit@10 : 0.1408
  mrr@5 : 0.0491
  mrr@10 : 0.0560

LightGCN sauvegardé ✅


In [1]:
import json, os
import pandas as pd

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'

# Charger les résultats des 4 modèles
all_results = {}
for model in ['BPRMF', 'NGCF', 'LightGCN', 'KGAT']:
    path = os.path.join(RESULTS_REC, f'{model}_results.json')
    if os.path.exists(path):
        with open(path, 'r') as f:
            all_results[model] = json.load(f)
        print(f'{model} chargé ✅')
    else:
        print(f'{model} — fichier introuvable ❌')

# Tableau comparatif
rows = []
for model, metrics in all_results.items():
    rows.append({
        'Modèle'      : model,
        'Recall@5'    : metrics.get('recall@5',    0),
        'Recall@10'   : metrics.get('recall@10',   0),
        'NDCG@5'      : metrics.get('ndcg@5',      0),
        'NDCG@10'     : metrics.get('ndcg@10',     0),
        'Precision@5' : metrics.get('precision@5', 0),
        'Precision@10': metrics.get('precision@10',0),
        'Hit@5'       : metrics.get('hit@5',       0),
        'Hit@10'      : metrics.get('hit@10',      0),
        'MRR@5'       : metrics.get('mrr@5',       0),
        'MRR@10'      : metrics.get('mrr@10',      0),
    })

df = pd.DataFrame(rows).set_index('Modèle')
df = df.sort_values('NDCG@10', ascending=False)

print('\n' + '='*70)
print('  Comparaison des modèles de recommandation')
print('='*70)
print(df.to_string())
print('='*70)

best_model = df.index[0]
print(f'\n🏆 Meilleur modèle : {best_model}')
print(f'   NDCG@10  : {df.loc[best_model, "NDCG@10"]:.4f}')
print(f'   Recall@10: {df.loc[best_model, "Recall@10"]:.4f}')
print(f'   MRR@10   : {df.loc[best_model, "MRR@10"]:.4f}')

# Sauvegarder
df.to_csv(os.path.join(RESULTS_REC, 'comparison_reco_models.csv'))
print(f'\nTableau sauvegardé ✅')

BPRMF chargé ✅
NGCF chargé ✅
LightGCN chargé ✅
KGAT chargé ✅

  Comparaison des modèles de recommandation
          Recall@5  Recall@10  NDCG@5  NDCG@10  Precision@5  Precision@10   Hit@5  Hit@10   MRR@5  MRR@10
Modèle                                                                                                   
BPRMF       0.1540     0.2292  0.1049   0.1291       0.0308        0.0229  0.1540  0.2292  0.0888  0.0988
KGAT        0.1511     0.2229  0.1017   0.1248       0.0302        0.0223  0.1511  0.2229  0.0855  0.0950
NGCF        0.1268     0.1942  0.0855   0.1071       0.0254        0.0194  0.1268  0.1942  0.0720  0.0808
LightGCN    0.0883     0.1408  0.0588   0.0756       0.0177        0.0141  0.0883  0.1408  0.0491  0.0560

🏆 Meilleur modèle : BPRMF
   NDCG@10  : 0.1291
   Recall@10: 0.2292
   MRR@10   : 0.0988

Tableau sauvegardé ✅


In [2]:
import glob, os

# Chercher le fichier .pth de BPRMF
searches = [
    r'D:\Memoire\LLMKGER\saved\BPR*.pth',
    r'saved\BPR*.pth',
    r'BPR*.pth',
]

found = []
for pattern in searches:
    files = glob.glob(pattern)
    found.extend(files)

if found:
    for f in found:
        size = os.path.getsize(f) / 1024 / 1024
        print(f'Trouvé : {f} ({size:.1f} MB)')
else:
    print('Aucun fichier BPR*.pth trouvé')
    # Chercher tous les .pth sauvegardés
    print('\nRecherche de tous les .pth :')
    for pattern in [r'D:\Memoire\LLMKGER\saved\*.pth', r'saved\*.pth']:
        files = glob.glob(pattern)
        for f in files:
            print(f'  {f}')

Trouvé : saved\BPR-Jun-25-2026_12-29-00.pth (22.2 MB)
Trouvé : saved\BPR-Jun-25-2026_12-59-14.pth (22.2 MB)


In [3]:
import torch
import pandas as pd
import numpy as np
import os
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model

RESULTS_REC  = r'D:\Memoire\LLMKGER\reco_results'
KG_DIR       = r'D:\Memoire\LLMKGER\kg_final'
RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data'

# Recharger la config BPRMF
base_config = {
    'data_path'        : RECBOLE_ROOT,
    'dataset'          : 'llmkger',
    'USER_ID_FIELD'    : 'user_id',
    'ITEM_ID_FIELD'    : 'item_id',
    'RATING_FIELD'     : 'rating',
    'load_col'         : {'inter': ['user_id', 'item_id', 'rating']},
    'eval_args'        : {'split': {'LS': 'valid_and_test'}, 'order': 'RO',
                          'group_by': 'user', 'mode': 'full'},
    'metrics'          : ['Recall', 'NDCG', 'Precision', 'Hit', 'MRR'],
    'topk'             : [5, 10],
    'valid_metric'     : 'NDCG@10',
    'epochs'           : 30,
    'train_batch_size' : 2048,
    'eval_batch_size'  : 2048,
    'learning_rate'    : 0.001,
    'embedding_size'   : 64,
    'stopping_step'    : 5,
    'device'           : 'cuda',
    'seed'             : 42,
    'show_progress'    : False,
}

config = Config(model='BPR', dataset='llmkger', config_dict=base_config)
init_seed(config['seed'], config['reproducibility'])
init_logger(config)

# Recréer le dataset pour avoir les mappings
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

print(f'Dataset chargé ✅')
print(f'Users : {dataset.num(config["USER_ID_FIELD"]):,}')
print(f'Items : {dataset.num(config["ITEM_ID_FIELD"]):,}')

# Charger le modèle BPRMF
model = get_model('BPR')(config, train_data._dataset).to(config['device'])
checkpoint = torch.load(
    r'saved\BPR-Jun-25-2026_12-59-14.pth',
    map_location=config['device'],
    weights_only=False
)
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print('Modèle BPRMF chargé ✅')

# Vérifier les embeddings
user_emb = model.user_embedding.weight
item_emb = model.item_embedding.weight
print(f'User embeddings : {user_emb.shape}')
print(f'Item embeddings : {item_emb.shape}')

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

Dataset chargé ✅
Users : 27,554
Items : 2,809
Modèle BPRMF chargé ✅
User embeddings : torch.Size([27554, 64])
Item embeddings : torch.Size([2809, 64])


In [4]:
from tqdm import tqdm
import torch.nn.functional as F

# Charger le test set
interactions_test = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_test.tsv'),
    sep='\t', header=0, names=['user_id', 'relation', 'course_id']
)
print(f'Test set : {len(interactions_test):,} paires')

# Récupérer les mappings RecBole (user_id string → index interne)
user2id = dataset.field2token_id[config['USER_ID_FIELD']]
item2id = dataset.field2token_id[config['ITEM_ID_FIELD']]

print(f'Users dans mapping : {len(user2id):,}')
print(f'Items dans mapping : {len(item2id):,}')

# Calculer S_emb pour chaque paire (user, cours)
s_emb_records = []
not_found = 0

with torch.no_grad():
    for _, row in tqdm(interactions_test.iterrows(),
                       total=len(interactions_test), desc='S_emb'):
        uid = row['user_id']
        cid = row['course_id']

        if uid not in user2id or cid not in item2id:
            s_emb_records.append({'user_id': uid, 'course_id': cid, 'S_emb': 0.0})
            not_found += 1
            continue

        u_idx = torch.tensor([user2id[uid]]).to(config['device'])
        i_idx = torch.tensor([item2id[cid]]).to(config['device'])

        u_vec = model.user_embedding(u_idx)
        i_vec = model.item_embedding(i_idx)

        score = torch.dot(u_vec.squeeze(), i_vec.squeeze()).item()
        s_emb_records.append({'user_id': uid, 'course_id': cid, 'S_emb': score})

df_semb = pd.DataFrame(s_emb_records)

print(f'\nS_emb calculé pour {len(df_semb):,} paires')
print(f'Paires hors mapping : {not_found:,}')
print(f'\nStatistiques S_emb :')
print(df_semb['S_emb'].describe())

# Sauvegarder
semb_path = os.path.join(RESULTS_REC, 'S_emb_scores_final.csv')
df_semb.to_csv(semb_path, index=False)
print(f'\nS_emb sauvegardé → {semb_path} ✅')

Test set : 106,735 paires
Users dans mapping : 27,554
Items dans mapping : 2,809


S_emb: 100%|██████████| 106735/106735 [00:12<00:00, 8877.31it/s]



S_emb calculé pour 106,735 paires
Paires hors mapping : 79,182

Statistiques S_emb :
count    106735.000000
mean          1.251288
std           2.314418
min          -7.728175
25%           0.000000
50%           0.000000
75%           1.703533
max          12.086021
Name: S_emb, dtype: float64

S_emb sauvegardé → D:\Memoire\LLMKGER\reco_results\S_emb_scores_final.csv ✅


In [5]:
# Comprendre le problème
users_test = set(interactions_test['user_id'].unique())
users_mapping = set(user2id.keys()) - {'[PAD]'}  # enlever token spécial

overlap = users_test & users_mapping
print(f'Users dans test set    : {len(users_test):,}')
print(f'Users dans mapping     : {len(users_mapping):,}')
print(f'Overlap                : {len(overlap):,}')
print(f'Users manquants        : {len(users_test - users_mapping):,}')

items_test = set(interactions_test['course_id'].unique())
items_mapping = set(item2id.keys()) - {'[PAD]'}
print(f'\nItems dans test set    : {len(items_test):,}')
print(f'Items dans mapping     : {len(items_mapping):,}')
print(f'Overlap items          : {len(items_test & items_mapping):,}')

Users dans test set    : 106,735
Users dans mapping     : 27,553
Overlap                : 27,553
Users manquants        : 79,182

Items dans test set    : 2,399
Items dans mapping     : 2,808
Overlap items          : 2,398


In [7]:
# Voir les champs disponibles dans le batch
for batch in test_data:
    interaction = batch[0]
    print('Champs disponibles :', list(interaction.interaction.keys()))
    break

Champs disponibles : ['user_id']


In [8]:
import torch
import pandas as pd
import numpy as np
import os

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'

# Récupérer tous les users et items du test set RecBole
id2user = dataset.id2token(config['USER_ID_FIELD'], 
                            list(range(dataset.num(config['USER_ID_FIELD']))))
id2item = dataset.id2token(config['ITEM_ID_FIELD'], 
                            list(range(dataset.num(config['ITEM_ID_FIELD']))))

print(f'Users dans modèle : {len(id2user):,}')
print(f'Items dans modèle : {len(id2item):,}')

# Extraire les users du test set
test_user_ids = []
for batch in test_data:
    interaction = batch[0]
    test_user_ids.extend(interaction['user_id'].tolist())

print(f'Users dans test RecBole : {len(test_user_ids):,}')

# Calculer S_emb pour chaque (user, item positif du test)
# On récupère l'item positif depuis les interactions originales
inter_test = pd.read_csv(
    os.path.join(r'D:\Memoire\LLMKGER\recbole_data\llmkger', 'llmkger.inter'),
    sep='\t'
)
print(f'\nColonnes .inter : {inter_test.columns.tolist()}')
print(inter_test.head(3))

Users dans modèle : 27,554
Items dans modèle : 2,809
Users dans test RecBole : 27,553

Colonnes .inter : ['user_id:token', 'item_id:token', 'rating:float']
  user_id:token item_id:token  rating:float
0    U_14092091      C_681678           1.0
1    U_14092091      C_629559           1.0
2    U_14092091      C_696549           1.0


In [9]:
from tqdm import tqdm

# Reconstruire le test set depuis le fichier .inter avec LOO
# (le dernier item de chaque user = test)
inter_path = r'D:\Memoire\LLMKGER\recbole_data\llmkger\llmkger.inter'
df_inter = pd.read_csv(inter_path, sep='\t')
df_inter.columns = ['user_id', 'item_id', 'rating']

# Dernier item de chaque user = test (LOO)
df_test_recbole = df_inter.groupby('user_id').last().reset_index()
print(f'Test set RecBole (LOO) : {len(df_test_recbole):,} paires')
print(df_test_recbole.head(3))

# Calculer S_emb
s_emb_records = []
not_found = 0

with torch.no_grad():
    for _, row in tqdm(df_test_recbole.iterrows(),
                       total=len(df_test_recbole), desc='S_emb'):
        uid = row['user_id']
        cid = row['item_id']

        if uid not in user2id or cid not in item2id:
            not_found += 1
            s_emb_records.append({'user_id': uid, 'course_id': cid, 'S_emb': 0.0})
            continue

        u_idx = torch.tensor([user2id[uid]]).to(config['device'])
        i_idx = torch.tensor([item2id[cid]]).to(config['device'])

        u_vec = model.user_embedding(u_idx)
        i_vec = model.item_embedding(i_idx)
        score = torch.dot(u_vec.squeeze(), i_vec.squeeze()).item()

        s_emb_records.append({'user_id': uid, 'course_id': cid, 'S_emb': score})

df_semb = pd.DataFrame(s_emb_records)
print(f'\nS_emb calculé : {len(df_semb):,} paires')
print(f'Paires hors mapping : {not_found:,}')
print(f'\nStatistiques S_emb :')
print(df_semb['S_emb'].describe())

semb_path = os.path.join(RESULTS_REC, 'S_emb_scores_final.csv')
df_semb.to_csv(semb_path, index=False)
print(f'\nS_emb sauvegardé → {semb_path} ✅')

Test set RecBole (LOO) : 27,553 paires
     user_id   item_id  rating
0   U_100090  C_676642     1.0
1  U_1000902  C_854866     1.0
2  U_1001872  C_697270     1.0


S_emb: 100%|██████████| 27553/27553 [00:08<00:00, 3205.28it/s]



S_emb calculé : 27,553 paires
Paires hors mapping : 0

Statistiques S_emb :
count    27553.000000
mean         4.847250
std          1.821971
min         -7.728175
25%          3.575366
50%          4.775087
75%          6.066682
max         12.086021
Name: S_emb, dtype: float64

S_emb sauvegardé → D:\Memoire\LLMKGER\reco_results\S_emb_scores_final.csv ✅


In [1]:
import json, os, glob

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'

print("=== Conditions d'entraînement de chaque modèle ===\n")

# Chercher les fichiers log RecBole pour voir epochs réels + config
for model in ['BPR', 'NGCF', 'LightGCN', 'KGAT']:
    logs = glob.glob(rf'D:\Memoire\LLMKGER\log\{model}\*.log')
    logs += glob.glob(rf'log\{model}\*.log')
    print(f'{model:10s} : {len(logs)} fichier(s) log trouvé(s)')
    for l in sorted(logs)[-1:]:
        print(f'           {l}')

=== Conditions d'entraînement de chaque modèle ===

BPR        : 3 fichier(s) log trouvé(s)
           log\BPR\BPR-llmkger-Jun-26-2026_03-48-31-55dc31.log
NGCF       : 2 fichier(s) log trouvé(s)
           log\NGCF\NGCF-llmkger-Jun-25-2026_13-28-42-79aa96.log
LightGCN   : 1 fichier(s) log trouvé(s)
           log\LightGCN\LightGCN-llmkger-Jun-25-2026_13-46-55-ddd272.log
KGAT       : 1 fichier(s) log trouvé(s)
           log\KGAT\KGAT-llmkger-Jun-25-2026_14-12-41-3e7dd3.log


In [2]:
import glob, re, os

def lire_log(model):
    logs = glob.glob(rf'log\{model}\*.log')
    if not logs:
        return None
    dernier = sorted(logs)[-1]
    with open(dernier, 'r', encoding='utf-8', errors='ignore') as f:
        contenu = f.read()
    return contenu

for model in ['BPR', 'NGCF', 'LightGCN', 'KGAT']:
    contenu = lire_log(model)
    if not contenu:
        print(f'{model} : pas de log\n')
        continue

    # epochs réels (dernier epoch atteint)
    epochs = re.findall(r'epoch (\d+) training', contenu)
    dernier_epoch = max(map(int, epochs)) if epochs else '?'

    # learning rate
    lr = re.search(r"'learning_rate':\s*([\d.]+)", contenu)
    lr = lr.group(1) if lr else '?'

    # train batch size
    bs = re.search(r"train_batch_size\s*=\s*\[?(\d+)", contenu)
    bs = bs.group(1) if bs else '?'

    # embedding size
    emb = re.search(r"'embedding_size':\s*(\d+)", contenu)
    emb = emb.group(1) if emb else '?'

    # stopping step
    stop = re.search(r"'stopping_step':\s*(\d+)", contenu)
    stop = stop.group(1) if stop else '?'

    # nombre de users / items vus
    nusers = re.search(r'The number of users:\s*(\d+)', contenu)
    nusers = nusers.group(1) if nusers else '?'
    nitems = re.search(r'The number of items:\s*(\d+)', contenu)
    nitems = nitems.group(1) if nitems else '?'

    print(f'=== {model} ===')
    print(f'  Dernier epoch atteint : {dernier_epoch}')
    print(f'  learning_rate         : {lr}')
    print(f'  train_batch_size      : {bs}')
    print(f'  embedding_size        : {emb}')
    print(f'  stopping_step         : {stop}')
    print(f'  users / items         : {nusers} / {nitems}')
    print()

=== BPR ===
  Dernier epoch atteint : ?
  learning_rate         : ?
  train_batch_size      : 2048
  embedding_size        : ?
  stopping_step         : ?
  users / items         : ? / ?

NGCF : pas de log

LightGCN : pas de log

KGAT : pas de log



In [3]:
import glob, os

# Lire le log BPR en entier (début + recherche des lignes clés)
log_bpr = sorted(glob.glob(r'log\BPR\*.log'))[-1]
print(f'Fichier : {log_bpr}\n')

with open(log_bpr, 'r', encoding='utf-8', errors='ignore') as f:
    lignes = f.readlines()

print(f'Nombre de lignes : {len(lignes)}\n')
print('=== 40 PREMIÈRES LIGNES (config) ===')
for l in lignes[:40]:
    print(l.rstrip())

Fichier : log\BPR\BPR-llmkger-Jun-26-2026_03-48-31-55dc31.log

Nombre de lignes : 2

=== 40 PREMIÈRES LIGNES (config) ===
Fri 26 Jun 2026 03:48:36 INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
Fri 26 Jun 2026 03:48:36 INFO  [Evaluation]: eval_batch_size = [2048] eval_args: [{'split': {'LS': 'valid_and_test'}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


In [4]:
import glob, os, re

def plus_gros_log(model):
    logs = glob.glob(rf'log\{model}\*.log')
    if not logs:
        return None
    # prendre le fichier le plus volumineux = l'entraînement complet
    return max(logs, key=os.path.getsize)

for model in ['BPR', 'NGCF', 'LightGCN', 'KGAT']:
    log = plus_gros_log(model)
    if not log:
        print(f'{model:9s} : aucun log\n')
        continue

    with open(log, 'r', encoding='utf-8', errors='ignore') as f:
        c = f.read()

    epochs = re.findall(r'epoch (\d+) training', c)
    dernier = max(map(int, epochs)) if epochs else '?'

    nusers = re.search(r'number of users[:\s]+(\d+)', c)
    nitems = re.search(r'number of items[:\s]+(\d+)', c)
    emb    = re.search(r'embedding_size["\s:=]+(\d+)', c)
    lr     = re.search(r'learning_rate["\s:=]+([\d.]+)', c)
    stop   = re.search(r'stopping_step["\s:=]+(\d+)', c)
    epoch_cfg = re.search(r"'epochs':\s*(\d+)|epochs\s*=\s*(\d+)", c)

    taille = os.path.getsize(log)/1024
    print(f'=== {model} ===  ({os.path.basename(log)}, {taille:.0f} Ko)')
    print(f'  users / items    : {nusers.group(1) if nusers else "?"} / {nitems.group(1) if nitems else "?"}')
    print(f'  embedding_size   : {emb.group(1) if emb else "?"}')
    print(f'  learning_rate    : {lr.group(1) if lr else "?"}')
    print(f'  stopping_step    : {stop.group(1) if stop else "?"}')
    print(f'  dernier epoch    : {dernier}')
    print()

=== BPR ===  (BPR-llmkger-Jun-25-2026_12-28-45-7b09b4.log, 62 Ko)
  users / items    : ? / ?
  embedding_size   : ?
  learning_rate    : ?
  stopping_step    : ?
  dernier epoch    : 29

=== NGCF ===  (NGCF-llmkger-Jun-25-2026_13-27-29-79aa96.log, 0 Ko)
  users / items    : ? / ?
  embedding_size   : ?
  learning_rate    : ?
  stopping_step    : ?
  dernier epoch    : ?

=== LightGCN ===  (LightGCN-llmkger-Jun-25-2026_13-46-55-ddd272.log, 0 Ko)
  users / items    : ? / ?
  embedding_size   : ?
  learning_rate    : ?
  stopping_step    : ?
  dernier epoch    : ?

=== KGAT ===  (KGAT-llmkger-Jun-25-2026_14-12-41-3e7dd3.log, 0 Ko)
  users / items    : ? / ?
  embedding_size   : ?
  learning_rate    : ?
  stopping_step    : ?
  dernier epoch    : ?



In [5]:
import os
p = r'D:\Memoire\LLMKGER\recbole_data_large\llmkger'
for f in os.listdir(p):
    taille = os.path.getsize(os.path.join(p, f)) / 1024 / 1024
    print(f'{f} : {taille:.1f} Mo')

llmkger.inter : 47.4 Mo
llmkger.kg : 40.4 Mo
llmkger.link : 0.1 Mo
